# Movie Industry Data Analysis and Clustering

**Portfolio version of a university data-analysis project**

This project explores a movie-industry dataset using exploratory data analysis and **K-means clustering**.

The main questions are:

1. How are production budget, IMDb score, rating category, and gross revenue related?
2. Which genres tend to have higher typical gross revenue?
3. Can movies be grouped into interpretable clusters based on financial and rating characteristics?

The analysis is descriptive and exploratory. It does **not** make causal claims about what causes a movie to succeed.

## 1. Setup

Install the required packages once if necessary:

```r
install.packages(c("tidyverse"))
```

In [ ]:
library(tidyverse)

theme_set(theme_minimal(base_size = 12))

## 2. Load and prepare the data

In [ ]:
data_path <- "data/movies.csv"

if (!file.exists(data_path)) {
  stop(
    paste(
      "movies.csv was not found.",
      "Download the Movie Industry dataset from Kaggle and place movies.csv in the data/ folder."
    )
  )
}

movies <- read_csv(data_path, show_col_types = FALSE)

glimpse(movies)

analysis_data <- movies |>
  transmute(
    name,
    year,
    genre,
    rating,
    score,
    budget,
    gross,
    r_rated = if_else(rating == "R", 1, 0)
  ) |>
  drop_na(score, rating, budget, gross)

tibble(
  original_rows = nrow(movies),
  analysis_rows = nrow(analysis_data),
  first_year = min(movies$year, na.rm = TRUE),
  last_year = max(movies$year, na.rm = TRUE)
)


Only observations with non-missing values for the variables used in the analysis are removed. This avoids discarding a movie merely because an unrelated field (for example, company or writer) is missing.

The binary `r_rated` variable is used only as a compact rating feature for clustering. It intentionally loses information contained in the full rating categories, which is a limitation discussed later.

## 3. Exploratory data analysis

In [ ]:
ggplot(analysis_data, aes(x = score, y = gross)) +
  geom_point(alpha = 0.25) +
  scale_y_log10(labels = scales::label_dollar()) +
  labs(
    title = "IMDb score and gross revenue",
    x = "IMDb score",
    y = "Gross revenue (log scale)"
  )

In [ ]:
ggplot(analysis_data, aes(x = budget, y = gross)) +
  geom_point(alpha = 0.25) +
  scale_x_log10(labels = scales::label_dollar()) +
  scale_y_log10(labels = scales::label_dollar()) +
  labs(
    title = "Production budget and gross revenue",
    x = "Budget (log scale)",
    y = "Gross revenue (log scale)"
  )

Gross revenue and production budget are strongly right-skewed, so logarithmic axes make the structure easier to inspect. The plots show association, not causation.

### Gross revenue by genre

In [ ]:
genre_summary <- analysis_data |>
  group_by(genre) |>
  summarise(
    movies = n(),
    median_gross = median(gross),
    mean_gross = mean(gross),
    median_budget = median(budget),
    .groups = "drop"
  ) |>
  filter(movies >= 30) |>
  arrange(desc(median_gross))

genre_summary

In [ ]:
genre_summary |>
  slice_max(median_gross, n = 10) |>
  ggplot(aes(x = reorder(genre, median_gross), y = median_gross)) +
  geom_col() +
  coord_flip() +
  scale_y_continuous(labels = scales::label_dollar()) +
  labs(
    title = "Genres with the highest median gross revenue",
    x = NULL,
    y = "Median gross revenue"
  )

Median gross revenue is reported alongside the mean because movie revenue is highly skewed and a small number of blockbusters can strongly affect averages.

## 4. Prepare features for clustering

In [ ]:
cluster_data <- analysis_data |>
  mutate(
    log_budget = log1p(budget),
    log_gross = log1p(gross)
  )

cluster_features <- cluster_data |>
  select(log_budget, log_gross, score, r_rated)

scaled_features <- scale(cluster_features)

K-means is sensitive to feature scale. The clustering variables are therefore standardized before fitting the model. Budget and gross revenue are log-transformed first to reduce the influence of extreme values.

## 5. Choose the number of clusters

In [ ]:
set.seed(9999)

k_values <- 1:10

wss <- map_dbl(k_values, function(k) {
  kmeans(
    scaled_features,
    centers = k,
    nstart = 50
  )$tot.withinss
})

selection_table <- tibble(
  k = k_values,
  WSS = wss
)

selection_table

ggplot(selection_table, aes(x = k, y = WSS)) +
  geom_line() +
  geom_point(size = 2) +
  scale_x_continuous(breaks = k_values) +
  labs(
    title = "Elbow plot for K-means",
    x = "Number of clusters",
    y = "Total within-cluster sum of squares"
  )

The elbow plot is an exploratory model-selection tool rather than a formal proof that one value of \(k\) is uniquely correct.

To keep the selection reproducible, the code below estimates the elbow as the point with the largest distance from the straight line joining the first and last points of the normalized WSS curve.

In [ ]:
x_norm <- (selection_table$k - min(selection_table$k)) /
          (max(selection_table$k) - min(selection_table$k))

y_norm <- (selection_table$WSS - min(selection_table$WSS)) /
          (max(selection_table$WSS) - min(selection_table$WSS))

distance_from_line <- (1 - x_norm) - y_norm

chosen_k <- selection_table$k[which.max(distance_from_line)]
cat("Selected number of clusters:", chosen_k, "\n")

## 6. Fit and profile the K-means model

In [ ]:
set.seed(9999)

movie_kmeans <- kmeans(
  scaled_features,
  centers = chosen_k,
  nstart = 50
)

clustered_movies <- cluster_data |>
  mutate(cluster = factor(movie_kmeans$cluster))

cluster_summary <- clustered_movies |>
  group_by(cluster) |>
  summarise(
    movies = n(),
    median_budget = median(budget),
    median_gross = median(gross),
    mean_score = mean(score),
    share_r_rated = mean(r_rated),
    .groups = "drop"
  ) |>
  arrange(desc(median_gross))

cluster_summary

In [ ]:
ggplot(
  clustered_movies,
  aes(x = budget, y = gross, color = cluster)
) +
  geom_point(alpha = 0.45) +
  scale_x_log10(labels = scales::label_dollar()) +
  scale_y_log10(labels = scales::label_dollar()) +
  labs(
    title = "Movie clusters by budget and gross revenue",
    x = "Budget (log scale)",
    y = "Gross revenue (log scale)",
    color = "Cluster"
  )

### Genre composition of each cluster

In [ ]:
cluster_genres <- clustered_movies |>
  count(cluster, genre, name = "movies") |>
  group_by(cluster) |>
  mutate(share = movies / sum(movies)) |>
  slice_max(share, n = 5, with_ties = FALSE) |>
  ungroup() |>
  arrange(cluster, desc(share))

cluster_genres

## 7. Interpretation

The cluster summary should be interpreted as a description of groups found by the algorithm, not as evidence that budget, rating, or IMDb score *causes* higher revenue.

The most useful comparisons are:

- differences in median budget and median gross revenue across clusters;
- whether high-revenue clusters also have higher IMDb scores;
- how the proportion of R-rated films varies between clusters;
- which genres are most common within each cluster.

Because gross revenue is itself included as a clustering feature, the clusters are designed partly around financial performance. They should therefore be viewed as a segmentation tool, not as a predictive model of future revenue.

## 8. Limitations

- K-means favors approximately spherical clusters in the feature space.
- The result depends on feature selection, transformations, scaling, and the chosen number of clusters.
- Reducing movie rating to `R` versus `non-R` discards information.
- Budget and gross figures may be inconsistently reported and are not inflation-adjusted.
- IMDb score is an audience rating, not an objective measure of quality.
- Associations in this dataset should not be interpreted causally.
- Revenue alone does not capture profitability because distribution, marketing, and other costs are not available.

## 9. Conclusion

This project demonstrates an exploratory data-analysis workflow combining data cleaning, visualization, feature transformation, standardization, and unsupervised learning.

The main value of K-means here is **segmentation**: it provides a compact way to compare groups of movies with different financial and rating profiles. The genre summaries complement the clustering analysis by showing how typical gross revenue varies across genres.

A natural next step would be to formulate a separate supervised-learning problem—for example, predicting log gross revenue or classifying films into financial-performance categories—and evaluate it on held-out data.

## Data source

Daniel Grijalva, *Movie Industry* dataset, Kaggle:  
https://www.kaggle.com/datasets/danielgrijalvas/movies

The dataset is not redistributed in this repository; see `data/README.md` for setup instructions.
